In [1]:
import pandas as pd

df = pd.read_csv("relevance_validation.csv")

df.head()

,uuid,title,maintext,jeremy_3way,julia_3way,jeremy_binary,julia_binary,consensus_3way
0,72d695f7-7ef5-4b3f-a1ed-fa9a4148cd21,"Putin running for reelection, almost sure to w...",Vladimir Putin on Friday moved to prolong his ...,0,0,0,0,0
1,1b25ea37-f2b8-452e-aec0-1e321235b05c,Proud Boys leader Enrique Tarrio found guilty ...,"Enrique Tarrio, leader of the Proud Boys, in O...",1,1,1,1,1
2,fdd46e8b-5263-49f8-8040-39abf87a89e3,Is The Rally Running Out Of Steam?,Gold prices are up 9% since February 14. (Phot...,0,0,0,0,0
3,a6ee042c-f271-4e17-8c78-5c4c705387fa,Google fires 28 employees after protest agains...,Protests across the U.S. call for Gaza cease-f...,2,2,1,1,2
4,a5160e01-1eae-4f6c-b52c-9fcd9bfcabff,Study finds 'no evidence' military veterans le...,Veterans are generally less likely than Americ...,1,1,1,1,1


In [2]:
df['consensus_binary'] = df["consensus_3way"].replace({2: 1})
df = df.drop(df.index[[21, 22, 23, 24]])
df['consensus_binary'].value_counts()

consensus_binary
1    27
0     9
Name: count, dtype: int64

In [44]:
import importlib
import relevance_prompts

importlib.reload(relevance_prompts)
PROMPT_TEXT = relevance_prompts.PROMPT

In [59]:
# Cell 1 — SETUP ONLY (binary relevance, Variant A, GPT-5) + JUSTIFICATION. Does NOT run any calls.

import os, re, json, time, random
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Import your Variant A prompt string from relevance_prompts.py

#from relevance_prompts import PROMPT as PROMPT_TEXT


# ---- PROMPT as dict (what run_* expects) ----
# IMPORTANT: Update your PROMPT_TEXT to require JSON output with BOTH keys:
# {"label": "RELEVANT"|"IRRELEVANT", "justification": "1-2 sentences ..."}
PROMPT_SPEC_A = {
    "name": "relevance_binary_variantA",
    "system": "You are a careful, literal classifier. Follow the rules and output format exactly. Output ONLY valid JSON.",
    "user_template": PROMPT_TEXT,  # expects {title} and {text}
}

# ---- LABELS (binary) ----
LABELS = ["IRRELEVANT", "RELEVANT"]

def normalize_gold_binary(consensus_binary) -> str:
    v = int(consensus_binary)
    return "IRRELEVANT" if v == 0 else "RELEVANT"

def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def normalize_pred_binary(label) -> Optional[str]:
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s or s in {"0", "false", "no"}:
        return "IRRELEVANT"
    if "rel" in s or s in {"1", "true", "yes"}:
        return "RELEVANT"
    return None

@dataclass
class GPTCallConfig:
    model: str = "gpt-5"
    temperature: float = 0.0  # not used by gpt-5; kept for compatibility
    max_output_tokens: int = 220  # slightly higher because we now want justification too
    max_retries: int = 5

def backoff_sleep(attempt: int):
    base = min(2 ** attempt, 30)
    time.sleep(base + random.uniform(0, 0.25 * base))

def _render_user_template(user_template: str, title: str, text: str) -> str:
    """
    Safe formatter:
    - Allows {title} and {text}
    - Escapes all other braces (so JSON examples in prompt don't break .format)
    """
    template = user_template.replace("{", "{{").replace("}", "}}")
    template = template.replace("{{title}}", "{title}").replace("{{text}}", "{text}")
    return template.format(title=title, text=text)

def _extract_response_text(resp) -> str:
    """
    Robust extraction for Responses API across models:
    - Prefer resp.output_text if present
    - Otherwise concatenate any text blocks in resp.output[*].content[*].text
    """
    if getattr(resp, "output_text", None):
        return (resp.output_text or "").strip()

    chunks = []
    try:
        for item in (resp.output or []):
            for c in (getattr(item, "content", None) or []):
                t = getattr(c, "text", None)
                if t:
                    chunks.append(t)
    except Exception:
        pass
    return "\n".join(chunks).strip()

def call_gpt_label_binary(client: OpenAI, prompt_spec: dict, title: str, text: str, cfg: GPTCallConfig):
    user_msg = _render_user_template(prompt_spec["user_template"], title=title, text=text)

    # Escalate output tokens across retries (helps when model "thinks" too long)
    token_budgets = [cfg.max_output_tokens, 300, 450]

    last_debug = ""
    for attempt in range(cfg.max_retries):
        try:
            max_toks = token_budgets[min(attempt, len(token_budgets)-1)]

            resp = client.responses.create(
                model=cfg.model,
                input=[
                    {"role": "system", "content": prompt_spec["system"]},
                    {"role": "user", "content": user_msg},
                ],
                reasoning={"effort": "low"},
                max_output_tokens=max_toks,
                # Optional: make it more likely to output the JSON quickly
                text={"verbosity": "low"},
            )

            out_text = _extract_response_text(resp)

            # Debug info (useful if it fails)
            status = getattr(resp, "status", None)
            incomplete = getattr(resp, "incomplete_details", None)
            usage = getattr(resp, "usage", None)
            last_debug = f"status={status} incomplete={incomplete} usage={usage}"

            # NEW: treat empty output or incomplete status as retryable failure
            if (not out_text) or (status and status != "completed"):
                # if we're at last attempt, return debug info in raw_output
                if attempt == cfg.max_retries - 1:
                    return None, "", f"EMPTY_OR_INCOMPLETE_RESPONSE | {last_debug}"
                continue

            # Parse JSON first (preferred)
            obj = extract_first_json_object(out_text)
            if obj and "label" in obj:
                pred = normalize_pred_binary(obj.get("label"))
                just = (obj.get("justification") or "").strip()
                if pred is not None:
                    return pred, just, out_text

            # Fallback: Label: 1/0
            m = re.search(r"label\s*:\s*([01])", out_text, flags=re.IGNORECASE)
            if m:
                pred = "RELEVANT" if m.group(1) == "1" else "IRRELEVANT"
                # attempt to grab any extra as justification
                just = ""
                parts = re.split(r"label\s*:\s*[01]\s*", out_text, flags=re.IGNORECASE, maxsplit=1)
                if len(parts) == 2:
                    just = parts[1].strip()
                return pred, just, out_text

            # Keyword fallback
            up = out_text.upper()
            if "RELEVANT" in up and "IRRELEVANT" not in up:
                return "RELEVANT", "", out_text
            if "IRRELEVANT" in up and "RELEVANT" not in up:
                return "IRRELEVANT", "", out_text

            # If parse failed, retry a couple times
            if attempt == cfg.max_retries - 1:
                return None, "", out_text

        except Exception as e:
            if attempt == cfg.max_retries - 1:
                return None, "", f"ERROR: {repr(e)}"
            backoff_sleep(attempt)

    return None, "", f"FAILED | {last_debug}"

def predict_one_row(df: pd.DataFrame, idx: int, client: OpenAI, cfg: GPTCallConfig, prompt_spec: dict) -> Dict[str, Any]:
    row = df.iloc[idx]
    pred, just, raw = call_gpt_label_binary(
        client=client,
        prompt_spec=prompt_spec,
        title=str(row["title"]),
        text=str(row["maintext"]),
        cfg=cfg
    )

    if "gold_label" in df.columns:
        gold = row["gold_label"]
    elif "consensus_binary" in df.columns:
        gold = normalize_gold_binary(row["consensus_binary"])
    else:
        gold = None

    return {
        "idx": idx,
        "pred": pred,
        "gold": gold,
        "justification": just,
        "raw_output": raw
    }

def run_binary_for_prompt(df: pd.DataFrame, prompt_spec: dict, client: OpenAI, cfg: GPTCallConfig) -> pd.DataFrame:
    preds, justs, raws = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=prompt_spec["name"]):
        p, j, raw = call_gpt_label_binary(
            client=client,
            prompt_spec=prompt_spec,
            title=str(row["title"]),
            text=str(row["maintext"]),
            cfg=cfg
        )
        preds.append(p)
        justs.append(j)
        raws.append(raw)

    out = df.copy()
    out["prompt_name"] = prompt_spec["name"]
    out["pred_label"] = preds
    out["justification"] = justs
    out["raw_output"] = raws
    out["is_valid_pred"] = out["pred_label"].isin(LABELS)
    return out

# ---- Setup client + config ----
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

cfg = GPTCallConfig(model="gpt-5", max_output_tokens=220, max_retries=5)

# ---- Ensure binary gold labels exist (does NOT call GPT) ----
if "gold_label" not in df.columns:
    if "consensus_binary" in df.columns:
        df["gold_label"] = df["consensus_binary"].apply(normalize_gold_binary)
    else:
        raise ValueError("Need df['gold_label'] (binary) or df['consensus_binary'] (0/1) to create it.")

print("Setup complete.")
print("Use: result = predict_one_row(df, idx, client, cfg, PROMPT_SPEC_A)")
print("Then inspect: result['pred'], result['gold'], result['justification'], result['raw_output']")
print("Use: bin_out = run_binary_for_prompt(df, PROMPT_SPEC_A, client, cfg)")

Setup complete.
Use: result = predict_one_row(df, idx, client, cfg, PROMPT_SPEC_A)
Then inspect: result['pred'], result['gold'], result['justification'], result['raw_output']
Use: bin_out = run_binary_for_prompt(df, PROMPT_SPEC_A, client, cfg)


In [63]:
# One-row test (example: first row)

result = predict_one_row(df, idx=10, client=client, cfg=cfg, prompt_spec=PROMPT_SPEC_A)

print("Pred:", result["pred"])
print("Gold:", result["gold"])
print("\nRAW OUTPUT:\n", result["raw_output"])

Pred: IRRELEVANT
Gold: IRRELEVANT

RAW OUTPUT:
 {
  "label": "IRRELEVANT",
  "justification": "The article centers on the Wagner Group’s armed mutiny and Kremlin negotiations, involving a militant mercenary group rather than civilian collective action. No protests, demonstrations, or civilian-led events are described."
}


In [ ]:
idx = 1
row = df.iloc[idx]

rendered_prompt = _render_user_template(
    PROMPT_SPEC_A["user_template"],
    title=str(row["title"]),
    text=str(row["maintext"])
)

print("===== START OF PROMPT =====\n")
print(rendered_prompt[:2000])  # first 2000 chars
print("\n...\n")
print("===== END OF PROMPT =====\n")
print(rendered_prompt[-2000:])  # last 2000 chars

print("\nLength of rendered prompt:", len(rendered_prompt))

In [ ]:
user_msg = _render_user_template(PROMPT_SPEC_A["user_template"], title=str(row["title"]), text=str(row["maintext"]))
print(user_msg[-1200:])  # print last ~1200 chars to see the TEXT section

In [64]:
# Cell 2 — OPTIONAL: run on all rows + save (run this only when ready)

bin_out = run_binary_for_prompt(df, PROMPT_SPEC_A, client, cfg)

os.makedirs("results", exist_ok=True)
bin_out.to_csv("results/relevance_binary_variantA.csv", index=False)

bin_out.head()

relevance_binary_variantA:   0%|          | 0/36 [00:00<?, ?it/s]

,uuid,title,maintext,jeremy_3way,julia_3way,jeremy_binary,julia_binary,consensus_3way,consensus_binary,gold_label,prompt_name,pred_label,justification,raw_output,is_valid_pred
0,72d695f7-7ef5-4b3f-a1ed-fa9a4148cd21,"Putin running for reelection, almost sure to w...",Vladimir Putin on Friday moved to prolong his ...,0,0,0,0,0,0,IRRELEVANT,relevance_binary_variantA,IRRELEVANT,The article focuses on Putin’s reelection bid ...,"{\n ""label"": ""IRRELEVANT"",\n ""justification""...",True
1,1b25ea37-f2b8-452e-aec0-1e321235b05c,Proud Boys leader Enrique Tarrio found guilty ...,"Enrique Tarrio, leader of the Proud Boys, in O...",1,1,1,1,1,1,RELEVANT,relevance_binary_variantA,RELEVANT,The article covers legal consequences stemming...,"{\n ""label"": ""RELEVANT"",\n ""justification"": ...",True
2,fdd46e8b-5263-49f8-8040-39abf87a89e3,Is The Rally Running Out Of Steam?,Gold prices are up 9% since February 14. (Phot...,0,0,0,0,0,0,IRRELEVANT,relevance_binary_variantA,IRRELEVANT,The article is a financial market commentary d...,"{\n ""label"": ""IRRELEVANT"",\n ""justification""...",True
3,a6ee042c-f271-4e17-8c78-5c4c705387fa,Google fires 28 employees after protest agains...,Protests across the U.S. call for Gaza cease-f...,2,2,1,1,2,1,RELEVANT,relevance_binary_variantA,RELEVANT,The article details specific employee protests...,"{\n ""label"": ""RELEVANT"",\n ""justification"": ...",True
4,a5160e01-1eae-4f6c-b52c-9fcd9bfcabff,Study finds 'no evidence' military veterans le...,Veterans are generally less likely than Americ...,1,1,1,1,1,1,RELEVANT,relevance_binary_variantA,RELEVANT,The article repeatedly references the specific...,"{\n ""label"": ""RELEVANT"",\n ""justification"": ...",True


In [65]:
# Cell 2 — Summary stats + Accuracy/F1 (binary)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

valid = bin_out[bin_out["is_valid_pred"]].copy()

y_true = valid["gold_label"].astype(str)
y_pred = valid["pred_label"].astype(str)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label="RELEVANT")

cm = confusion_matrix(y_true, y_pred, labels=LABELS)

print("N total:", len(bin_out))
print("N valid preds:", len(valid))
print("Invalid/None preds:", len(bin_out) - len(valid))
print("\nAccuracy:", acc)
print("F1 (RELEVANT as positive):", f1)
print("\nConfusion matrix (rows=true, cols=pred; labels=IRRELEVANT, RELEVANT):\n", cm)
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, labels=LABELS))

# Quick distribution sanity checks
print("\nGold label distribution:\n", y_true.value_counts(dropna=False))
print("\nPred label distribution:\n", y_pred.value_counts(dropna=False))

N total: 36
N valid preds: 36
Invalid/None preds: 0

Accuracy: 0.8611111111111112
F1 (RELEVANT as positive): 0.912280701754386

Confusion matrix (rows=true, cols=pred; labels=IRRELEVANT, RELEVANT):
 [[ 5  4]
 [ 1 26]]

Classification report:

              precision    recall  f1-score   support

  IRRELEVANT       0.83      0.56      0.67         9
    RELEVANT       0.87      0.96      0.91        27

    accuracy                           0.86        36
   macro avg       0.85      0.76      0.79        36
weighted avg       0.86      0.86      0.85        36


Gold label distribution:
 gold_label
RELEVANT      27
IRRELEVANT     9
Name: count, dtype: int64

Pred label distribution:
 pred_label
RELEVANT      30
IRRELEVANT     6
Name: count, dtype: int64
